In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("./neo4j_dump_20240819")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from octopus.client import OctopusClient

In [3]:
from getpass import getpass

username = input("Username:")
password = getpass("Password:")

Username: jihun
Password: ········


In [5]:
oc = OctopusClient(username, password)

## Get nodes data

In [6]:
execution = oc.execute("MATCH (n:entity) RETURN n")
execution.poll()

node_batches = []
for batch in execution.output:
    node_batches.append(batch)

In [6]:
nodes_path = OUTPUT_DIR / "nodes.csv"

In [8]:
import csv

In [9]:
nodes_header = ["id", "ocid", "name"]
with open(nodes_path, "w") as f:
    writer = csv.writer(f)
    writer.writerow(nodes_header)

In [10]:
with open(nodes_path, "a") as f:
    writer = csv.writer(f)
    for batch in node_batches:
        rows = batch["n"]
        for row in rows:
            writer.writerow([row["<id>"], row["ocid"], row["name"]])

## Get edges data

In [11]:
execution = oc.execute("MATCH (n1:entity)-[r]->(n2:entity) RETURN r")
execution.poll()

0

In [12]:
edge_batches = []
for batch in execution.output:
    edge_batches.append(batch)

In [19]:
edges_path = OUTPUT_DIR / "edges.csv"

In [22]:
edges_header = [
    "type",
    "start_id",
    "end_id",
    "ocid_relation",
    "date",
    "timestamp",
    "number_of_occurrences",
]
with open(edges_path, "w") as f:
    writer = csv.writer(f)
    writer.writerow(edges_header)

In [23]:
from tqdm.auto import tqdm

erroneous_rel_ids = []
with open(edges_path, "a") as f:
    writer = csv.writer(f)
    for batch in tqdm(edge_batches):
        rows = batch["r"]
        for row in rows:
            writer.writerow(
                [
                    row["<rel.type>"],
                    row["<source.id>"],
                    row["<target.id>"],
                    row["ocidRelation"],
                    row["date"],
                    row["timestamp"],
                    row["numberOfOccurrences"],
                ]
            )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [14:56<00:00, 18.69s/it]


## Sample nodes

In [19]:
SAMPLE_DIR = OUTPUT_DIR / "sampled_node_ocids"
SAMPLE_DIR.mkdir(exist_ok=True, parents=True)

In [9]:
node_ocids = set()
with open(nodes_path) as f:
    reader = csv.DictReader(f)
    for row in reader:
        node_ocids.add(row["ocid"])

In [10]:
len(node_ocids)

616700

In [11]:
import random

In [14]:
random.seed(2024)
node_ocids_rand = list(node_ocids)
random.shuffle(node_ocids_rand)

In [16]:
sample_sizes = [10, 100, 1000, 10000]

In [20]:
for sample_size in sample_sizes:
    sample_filename = f"random_node_ocids_{sample_size}.txt"
    with open(SAMPLE_DIR / sample_filename, "w") as f:
        for node_ocid in node_ocids_rand[:sample_size]:
            f.write(node_ocid)
            f.write("\n")